# ClinicalRecall — A100 GPU Ingestion on Colab

**Prerequisites — upload these to Google Drive at `MyDrive/ClinicalRecall/`:**
```
MyDrive/ClinicalRecall/
├── data/
│   ├── fhir/                  (5,770 JSON files)
│   ├── clinician_notes/       (5,770 JSON files)
│   └── mtsamples_staging.db
├── scripts/
│   ├── ingest_chromadb.py
│   └── ingest_chromadb_gpu.py
└── backend/
    └── ingestion/
        └── synthea_parser.py
```

**What this notebook does:**
1. Mounts Google Drive
2. Installs dependencies
3. Auto-tunes batch size for A100 (40GB VRAM)
4. Runs `--only synthea` ingestion
5. Copies the resulting `chroma_db/` back to Drive

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/ClinicalRecall/
!ls /content/drive/MyDrive/ClinicalRecall/data/
!ls /content/drive/MyDrive/ClinicalRecall/scripts/
!ls /content/drive/MyDrive/ClinicalRecall/backend/ingestion/

## 2. Install Dependencies

In [ ]:
!pip install -q chromadb sentence-transformers torch openai python-dotenv

## 3. Set Up Working Directory

In [ ]:
import os
import sys

DRIVE = '/content/drive/MyDrive/ClinicalRecall'
WORK  = '/content/ClinicalRecall'

os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

# Symlink data (avoid copying 18GB)
!ln -sfn {DRIVE}/data {WORK}/data

# Copy scripts and backend (small files)
!cp -r {DRIVE}/scripts {WORK}/scripts
!cp -r {DRIVE}/backend {WORK}/backend

# Add scripts to path
sys.path.insert(0, f'{WORK}/scripts')

print('Working directory:', os.getcwd())
!ls -la {WORK}/
!ls {WORK}/scripts/
!ls {WORK}/backend/ingestion/

## 4. Verify GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 5. Run Ingestion

Auto-tunes batch size for the A100, then ingests all 5,770 patients.
Expected: ~8-12 minutes on A100.

In [ ]:
!python scripts/ingest_chromadb_gpu.py --only synthea

## 6. Verify Output

In [ ]:
import sqlite3
conn = sqlite3.connect('chroma_db/chroma.sqlite3')

cols = conn.execute("SELECT name FROM collections").fetchall()
print(f"Collections: {[c[0] for c in cols]}")

total = conn.execute("SELECT COUNT(*) FROM embeddings").fetchone()[0]
print(f"Total embeddings: {total:,}")

patients = conn.execute('''
    SELECT COUNT(DISTINCT em.string_value) 
    FROM embedding_metadata em 
    WHERE em.key = 'patient_id'
''').fetchone()[0]
print(f"Unique patients: {patients:,}")

with_enc = conn.execute('''
    SELECT COUNT(DISTINCT em.id) 
    FROM embedding_metadata em 
    WHERE em.key = 'encounter_type' AND em.string_value != ''
''').fetchone()[0]
print(f"Chunks with encounter_type: {with_enc:,} ({with_enc/total*100:.1f}%)")

keys = conn.execute("SELECT DISTINCT key FROM embedding_metadata").fetchall()
print(f"Metadata keys: {sorted([k[0] for k in keys])}")

conn.close()

In [ ]:
!du -sh chroma_db/

## 7. Copy chroma_db to Drive

In [ ]:
!cp -r /content/ClinicalRecall/chroma_db /content/drive/MyDrive/ClinicalRecall/chroma_db_a100
print("Done! chroma_db copied to Drive as chroma_db_a100")